In [1]:
! pip install pandas
print("Pandas installed successfully.")

Pandas installed successfully.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [3]:
dataset_folder_path = "A:\\FlipKart_GridLock\\dataset"
import os
for filename in os.listdir(dataset_folder_path):
    if filename == "train.csv":
        train_csv_path = os.path.join(dataset_folder_path, filename)
    if filename == "test.csv":
        test_csv_path = os.path.join(dataset_folder_path, filename)

print(train_csv_path)
print(test_csv_path)            

A:\FlipKart_GridLock\dataset\train.csv
A:\FlipKart_GridLock\dataset\test.csv


In [4]:
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

In [5]:
print("Dataset Head:")
print(train_df.head(20))

print("Dataset Info:")
print(train_df.info())

print("Dataset Description:")
print(train_df.describe(include='all'))

print("Null Values:")
print(train_df.isnull().sum())

print("Unique Values:")
print(train_df.nunique())

Dataset Head:
    Index geohash  day timestamp    demand     RoadType  NumberofLanes  \
0       0  qp02z1   48       0:0  0.048804          NaN              1   
1       1  qp02zt   48       0:0  0.118507  Residential              3   
2       2  qp08bj   48       0:0  0.027132  Residential              1   
3       3  qp08gt   48       0:0  0.003272  Residential              1   
4       4  qp02zq   48       0:0  0.010819  Residential              1   
5       5  qp02zw   48       0:0  0.016262  Residential              2   
6       6  qp02zy   48       0:0  0.042247  Residential              3   
7       7  qp08by   48       0:0  0.040135  Residential              1   
8       8  qp08gq   48       0:0  0.001002  Residential              1   
9       9  qp08gy   48       0:0  0.003948  Residential              3   
10     10  qp02zp   48       0:0  0.017780  Residential              1   
11     11  qp02zr   48       0:0  0.089060  Residential              2   
12     12  qp02zx   48  

In [6]:
print(train_df.groupby("geohash")["demand"].agg(["mean", "std"]).head())
print(train_df.groupby("timestamp")["demand"].mean())
print(train_df.groupby(["geohash", "timestamp"])["demand"].mean().head(20))

             mean       std
geohash                    
qp02yc   0.018498  0.012374
qp02yf   0.029433       NaN
qp02yy   0.002902  0.001435
qp02yz   0.036564  0.022065
qp02z1   0.040048  0.030124
timestamp
0:0     0.081056
0:15    0.081929
0:30    0.084357
0:45    0.085994
10:0    0.110319
          ...   
8:45    0.105793
9:0     0.107004
9:15    0.112049
9:30    0.109178
9:45    0.108777
Name: demand, Length: 96, dtype: float64
geohash  timestamp
qp02yc   10:30        0.046790
         10:45        0.021158
         1:0          0.005397
         2:30         0.012944
         2:45         0.025961
         3:30         0.026422
         6:30         0.008387
         6:45         0.020780
         7:0          0.004853
         7:15         0.027092
         7:30         0.004824
         9:0          0.017362
qp02yf   11:15        0.029433
qp02yy   4:15         0.003916
         8:45         0.001887
qp02yz   10:0         0.069452
         10:15        0.061226
         10:30      

In [7]:
print(test_df["geohash"].nunique())

print(set(test_df["geohash"]) - set(train_df["geohash"]))

1190
{'qp09tv', 'qp09j5', 'qp09vh', 'qp09y0', 'qp091d', 'qp091n', 'qp0dn1', 'qp08g4', 'qp08ch', 'qp0965'}


## Note --> We'll not use Label encoding as primary spatial representation as unseen geohashes exist and Label encoding fails on unseen categories

# TimeStamp Parsaing


In [8]:
train_df[['hour', 'minute']] = train_df['timestamp'].str.split(':', expand=True).astype(int)

test_df[['hour', 'minute']] = test_df['timestamp'].str.split(':', expand=True).astype(int)

train_df['total_minutes'] = train_df['hour'] * 60 + train_df['minute']

test_df['total_minutes'] = test_df['hour'] * 60 + test_df['minute']

In [9]:
print(train_df[['timestamp', 'hour', 'minute', 'total_minutes']].head(10))

  timestamp  hour  minute  total_minutes
0       0:0     0       0              0
1       0:0     0       0              0
2       0:0     0       0              0
3       0:0     0       0              0
4       0:0     0       0              0
5       0:0     0       0              0
6       0:0     0       0              0
7       0:0     0       0              0
8       0:0     0       0              0
9       0:0     0       0              0


In [10]:
# Here we will create cyclic time features for hour and minute to capture the cyclical nature of time.
import numpy as np

train_df['time_sin'] = np.sin(2 * np.pi * train_df['total_minutes'] / 1440)
train_df['time_cos'] = np.cos(2 * np.pi * train_df['total_minutes'] / 1440)

test_df['time_sin'] = np.sin(2 * np.pi * test_df['total_minutes'] / 1440)
test_df['time_cos'] = np.cos(2 * np.pi * test_df['total_minutes'] / 1440)

In [11]:
print(train_df[['timestamp', 'time_sin', 'time_cos']].head())

  timestamp  time_sin  time_cos
0       0:0       0.0       1.0
1       0:0       0.0       1.0
2       0:0       0.0       1.0
3       0:0       0.0       1.0
4       0:0       0.0       1.0


### Decoding Geohash values into Latitude and longitude

In [12]:
! pip install pygeohash


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import pygeohash as pgh

train_df[['latitude', 'longitude']] = train_df['geohash'].apply(
    lambda x: pd.Series(pgh.decode(x))
)

test_df[['latitude', 'longitude']] = test_df['geohash'].apply(
    lambda x: pd.Series(pgh.decode(x))
)

In [14]:
print(train_df[['geohash', 'latitude', 'longitude']].head())

  geohash  latitude  longitude
0  qp02z1 -5.484924  90.664673
1  qp02zt -5.462952  90.686646
2  qp08bj -5.462952  90.708618
3  qp08gt -5.462952  90.862427
4  qp02zq -5.457458  90.675659


In [15]:
# Creating geohash hierarchy features
for i in [2, 3, 4, 5]:
    train_df[f'geo_{i}'] = train_df['geohash'].str[:i]
    test_df[f'geo_{i}'] = test_df['geohash'].str[:i]
    

print(train_df[['geohash', 'geo_2', 'geo_3', 'geo_4', 'geo_5']].head())    

  geohash geo_2 geo_3 geo_4  geo_5
0  qp02z1    qp   qp0  qp02  qp02z
1  qp02zt    qp   qp0  qp02  qp02z
2  qp08bj    qp   qp0  qp08  qp08b
3  qp08gt    qp   qp0  qp08  qp08g
4  qp02zq    qp   qp0  qp02  qp02z


here it represents the region based on area                         qp      → very broad region
qp0     → smaller region
qp02    → more localized
qp02z   → neighborhood-level
qp02z1  → precise location                                                                                            so we are doing this because test contains unseen geohashes So if the model only memorizes full geohash IDs, it struggles on unseen locations but hierarchical prefixes help the model generalize spatially.

In [16]:
# Missing Value indicator
train_df['temp_missing'] = train_df['Temperature'].isnull().astype(int)
test_df['temp_missing'] = test_df['Temperature'].isnull().astype(int)

train_df['weather_missing'] = train_df['Weather'].isnull().astype(int)
test_df['weather_missing'] = test_df['Weather'].isnull().astype(int)

train_df['road_missing'] = train_df['RoadType'].isnull().astype(int)
test_df['road_missing'] = test_df['RoadType'].isnull().astype(int)

In [17]:
print(train_df[['Temperature', 'temp_missing',
                'Weather', 'weather_missing',
                'RoadType', 'road_missing']].head())

   Temperature  temp_missing Weather  weather_missing     RoadType  \
0          NaN             1     NaN                1          NaN   
1    31.104565             0   Sunny                0  Residential   
2    25.919267             0   Sunny                0  Residential   
3          NaN             1   Rainy                0  Residential   
4    10.803667             0   Rainy                0  Residential   

   road_missing  
0             1  
1             0  
2             0  
3             0  
4             0  


### We are doing so  because Now the model can learn patterns like:
* missing weather data happens in certain regions
* sensor failures correlate with low/high demand
* incomplete records behave differently

If you directly imputed without flags, that information would be lost.

In [18]:
# Null value imputation
train_df['Temperature'] = train_df['Temperature'].fillna(train_df['Temperature'].median())
test_df['Temperature'] = test_df['Temperature'].fillna(test_df['Temperature'].median())

train_df['Weather'] = train_df['Weather'].fillna('Unknown')
test_df['Weather'] = test_df['Weather'].fillna('Unknown')

train_df['RoadType'] = train_df['RoadType'].fillna('Unknown')
test_df['RoadType'] = test_df['RoadType'].fillna('Unknown')

In [19]:
print(train_df.isnull().sum())

Index              0
geohash            0
day                0
timestamp          0
demand             0
RoadType           0
NumberofLanes      0
LargeVehicles      0
Landmarks          0
Temperature        0
Weather            0
hour               0
minute             0
total_minutes      0
time_sin           0
time_cos           0
latitude           0
longitude          0
geo_2              0
geo_3              0
geo_4              0
geo_5              0
temp_missing       0
weather_missing    0
road_missing       0
dtype: int64


In [20]:
! pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    'geohash',
    'geo_2',
    'geo_3',
    'geo_4',
    'geo_5',
    'RoadType',
    'Weather',
    'LargeVehicles',
    'Landmarks'
]

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()

    combined = pd.concat([train_df[col], test_df[col]], axis=0)

    le.fit(combined)

    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])

    label_encoders[col] = le

In [22]:
print(train_df[categorical_cols].head())

   geohash  geo_2  geo_3  geo_4  geo_5  RoadType  Weather  LargeVehicles  \
0        4      0      0      0      1         3        4              1   
1       25      0      0      0      1         1        3              0   
2      370      0      0      3     16         1        3              1   
3      418      0      0      3     19         1        1              1   
4       22      0      0      0      1         1        1              1   

   Landmarks  
0          0  
1          1  
2          0  
3          0  
4          0  


In [23]:
# feature selection for model
drop_cols = ['Index', 'timestamp', 'demand']

X = train_df.drop(columns=drop_cols)
y = train_df['demand']

X_test = test_df.drop(columns=['Index', 'timestamp']) 

In [24]:
print(X.shape)
print(X_test.shape)
print(X.columns)

(77299, 22)
(41778, 22)
Index(['geohash', 'day', 'RoadType', 'NumberofLanes', 'LargeVehicles',
       'Landmarks', 'Temperature', 'Weather', 'hour', 'minute',
       'total_minutes', 'time_sin', 'time_cos', 'latitude', 'longitude',
       'geo_2', 'geo_3', 'geo_4', 'geo_5', 'temp_missing', 'weather_missing',
       'road_missing'],
      dtype='str')


In [25]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

groups = train_df['geohash']

Instead of random splitting,
it ensures same geohash does not appear in both train and validation fold

Why this matters:
otherwise model memorizes location demand.
That gives fake CV scores.

In [26]:
# base model
! pip install lightgbm



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse'
    )

    val_preds = model.predict(X_valid)
    test_fold_preds = model.predict(X_test)

    oof_preds[valid_idx] = val_preds
    test_preds += test_fold_preds / 5

    rmse = np.sqrt(mean_squared_error(y_valid, val_preds))
    r2 = r2_score(y_valid, val_preds)

    print(f"Fold {fold+1}")
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-"*40)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007560 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 955
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 20
[LightGBM] [Info] Start training from score 0.095569
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

In [28]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

print(feature_importance)

            feature  importance
0           geohash       15598
13         latitude       11300
14        longitude        9777
6       Temperature        4871
10    total_minutes        4572
12         time_cos        3674
11         time_sin        3234
2          RoadType        2022
8              hour        1678
18            geo_5        1606
3     NumberofLanes        1354
9            minute         752
7           Weather         660
5         Landmarks         589
1               day         503
4     LargeVehicles         457
17            geo_4         214
19     temp_missing          96
21     road_missing          37
20  weather_missing           3
16            geo_3           0
15            geo_2           0


## Few Observations-->
* Raw geohash is dominating that means  llocation identity is the strongest signal , model is heavily memorizing so it may lead to overfitting 

* Latitude and Longitutde are extremely strong that means spatial continuity exists , model is learning geography 

* GeoHash hierarchy result working well as geo_5 is strong , geo_4 weak and geo_3 , geo_2 useless This tells us:only local neighborhoods matters , broad regional grouping is too coarse
So: we should probably DROP:geo_2 & geo_3



In [29]:
# Leakage safe target encoding

from sklearn.model_selection import KFold

train_df['geo_target_mean'] = 0.0
test_df['geo_target_mean'] = 0.0

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(train_df):

    X_train_fold = train_df.iloc[train_idx]
    X_val_fold = train_df.iloc[val_idx]

    geo_means = X_train_fold.groupby('geohash')['demand'].mean()

    train_df.loc[val_idx, 'geo_target_mean'] = (
        X_val_fold['geohash']
        .map(geo_means)
    )

global_mean = train_df['demand'].mean()

train_df['geo_target_mean'] = (
    train_df['geo_target_mean']
    .fillna(global_mean)
)

test_geo_means = train_df.groupby('geohash')['demand'].mean()

test_df['geo_target_mean'] = (
    test_df['geohash']
    .map(test_geo_means)
    .fillna(global_mean)
) 

In [30]:
print(train_df[['geohash', 'demand', 'geo_target_mean']].head())

   geohash    demand  geo_target_mean
0        4  0.048804         0.035970
1       25  0.118507         0.200072
2      370  0.027132         0.126477
3      418  0.003272         0.014768
4       22  0.010819         0.030562


the feature(geohash=25
actual demand = 0.118
geo_target_mean = 0.200) tells historically this region tends to have higher traffic 

In [31]:
# Retrain model with target encoding
drop_cols = ['Index', 'timestamp', 'demand']

X = train_df.drop(columns=drop_cols)
y = train_df['demand']

X_test = test_df.drop(columns=['Index', 'timestamp'])

In [32]:
! pip install matplotlib


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
! pip install xgboost
! pip install catboost


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# =====================================================
# GROUP KFOLD
# =====================================================

gkf = GroupKFold(n_splits=5)

# =====================================================
# REUSABLE TRAINING FUNCTION
# =====================================================

def train_model_cv(model, model_name, X, y, X_test, groups):

    # -----------------------------
    # STORAGE
    # -----------------------------

    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    rmse_scores = []
    train_r2_scores = []
    valid_r2_scores = []
    overfit_gaps = []

    feature_importance_df = pd.DataFrame()

    # =================================================
    # CV LOOP
    # =================================================

    for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups)):

        print(f"\n================ FOLD {fold+1} ================")

        # ---------------------------------------------
        # SPLIT
        # ---------------------------------------------

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        # ---------------------------------------------
        # TRAIN
        # ---------------------------------------------

        if model_name == "catboost":

            model.fit(
                X_train,
                y_train,
                eval_set=(X_valid, y_valid),
                verbose=0
            )

        else:

            model.fit(
                X_train,
                y_train
            )

        # ---------------------------------------------
        # PREDICTIONS
        # ---------------------------------------------

        val_preds = model.predict(X_valid)

        train_preds = model.predict(X_train)

        test_fold_preds = model.predict(X_test)

        # ---------------------------------------------
        # STORE OOF
        # ---------------------------------------------

        oof_preds[valid_idx] = val_preds

        # ---------------------------------------------
        # STORE TEST PREDS
        # ---------------------------------------------

        test_preds += test_fold_preds / 5

        # ---------------------------------------------
        # METRICS
        # ---------------------------------------------

        rmse = np.sqrt(mean_squared_error(y_valid, val_preds))

        train_r2 = r2_score(y_train, train_preds)

        valid_r2 = r2_score(y_valid, val_preds)

        gap = train_r2 - valid_r2

        # ---------------------------------------------
        # SAVE METRICS
        # ---------------------------------------------

        rmse_scores.append(rmse)

        train_r2_scores.append(train_r2)

        valid_r2_scores.append(valid_r2)

        overfit_gaps.append(gap)

        # ---------------------------------------------
        # PRINT
        # ---------------------------------------------

        print("RMSE:", rmse)

        print("Train R2:", train_r2)

        print("Validation R2:", valid_r2)

        print("Overfitting Gap:", gap)

        # ---------------------------------------------
        # FEATURE IMPORTANCE
        # ---------------------------------------------

        if hasattr(model, "feature_importances_"):

            fold_importance = pd.DataFrame({
                "feature": X.columns,
                "importance": model.feature_importances_
            })

            feature_importance_df = pd.concat(
                [feature_importance_df, fold_importance],
                axis=0
            )

    # =================================================
    # FINAL RESULTS
    # =================================================

    print(f"\n================ {model_name.upper()} FINAL RESULTS ================")

    print("Average RMSE:", np.mean(rmse_scores))

    print("Average Train R2:", np.mean(train_r2_scores))

    print("Average Validation R2:", np.mean(valid_r2_scores))

    print("Average Overfitting Gap:", np.mean(overfit_gaps))

    # =================================================
    # FEATURE IMPORTANCE
    # =================================================

    if not feature_importance_df.empty:

        importance = (
            feature_importance_df
            .groupby("feature")["importance"]
            .mean()
            .sort_values(ascending=False)
        )

        print("\nTop Features:\n")

        print(importance.head(20))

    # =================================================
    # RETURN EVERYTHING
    # =================================================

    return {

        "oof_preds": oof_preds,

        "test_preds": test_preds,

        "rmse": np.mean(rmse_scores),

        "train_r2": np.mean(train_r2_scores),

        "valid_r2": np.mean(valid_r2_scores),

        "gap": np.mean(overfit_gaps)
    }

In [35]:
lgbm_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

lgbm_results = train_model_cv(
    model=lgbm_model,
    model_name="lightgbm",
    X=X,
    y=y,
    X_test=X_test,
    groups=groups
)
oof_lgb = lgbm_results["oof_preds"]
test_lgb = lgbm_results["test_preds"]


================ FOLD 1 ================
RMSE: 0.03802578967565536
Train R2: 0.9698567531504192
Validation R2: 0.8949756240376742
Overfitting Gap: 0.07488112911274503

================ FOLD 2 ================
RMSE: 0.04606898909766797
Train R2: 0.9685887837655608
Validation R2: 0.8931749941009609
Overfitting Gap: 0.07541378966459988

================ FOLD 3 ================
RMSE: 0.038115040602913756
Train R2: 0.9685981686389681
Validation R2: 0.917417787465482
Overfitting Gap: 0.05118038117348611

================ FOLD 4 ================
RMSE: 0.04518134700183693
Train R2: 0.967055864032631
Validation R2: 0.9107277885738522
Overfitting Gap: 0.056328075458778826

================ FOLD 5 ================
RMSE: 0.04403932560994054
Train R2: 0.9655246819710289
Validation R2: 0.9278008119669349
Overfitting Gap: 0.03772387000409405

================ LIGHTGBM FINAL RESULTS ================
Average RMSE: 0.04228609839760291
Average Train R2: 0.9679248503117217
Average Validation R2: 0.908819

In [36]:
xgb_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

xgb_results = train_model_cv(
    model=xgb_model,
    model_name="xgboost",
    X=X,
    y=y,
    X_test=X_test,
    groups=groups
)
oof_xgb = xgb_results["oof_preds"]
test_xgb = xgb_results["test_preds"]


================ FOLD 1 ================
RMSE: 0.036479042496530514
Train R2: 0.9818065667554385
Validation R2: 0.9033458541879001
Overfitting Gap: 0.0784607125675384

================ FOLD 2 ================
RMSE: 0.04534687139560228
Train R2: 0.9806558644704307
Validation R2: 0.8964976479823887
Overfitting Gap: 0.084158216488042

================ FOLD 3 ================
RMSE: 0.03773790771904288
Train R2: 0.9811250668538781
Validation R2: 0.9190439374857059
Overfitting Gap: 0.06208112936817223

================ FOLD 4 ================
RMSE: 0.04614996881627676
Train R2: 0.9799413878277502
Validation R2: 0.9068590275456335
Overfitting Gap: 0.07308236028211668

================ FOLD 5 ================
RMSE: 0.04463111045019946
Train R2: 0.9789417319985109
Validation R2: 0.9258474007661084
Overfitting Gap: 0.05309433123240248

================ XGBOOST FINAL RESULTS ================
Average RMSE: 0.04206898017553038
Average Train R2: 0.9804941235812017
Average Validation R2: 0.910318773

In [37]:
from catboost import CatBoostRegressor

cat_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    
    l2_leaf_reg=10,
    
    subsample=0.8,
    
    loss_function='RMSE',
    
    eval_metric='RMSE',
    
    random_seed=42,
    
    verbose=200
)

cat_results = train_model_cv(
    model=cat_model,
    model_name="catboost",
    X=X,
    y=y,
    X_test=X_test,
    groups=groups
)
oof_cat = cat_results["oof_preds"]
test_cat = cat_results["test_preds"]


================ FOLD 1 ================
RMSE: 0.035219445783270985
Train R2: 0.9641529720212869
Validation R2: 0.9099054207579489
Overfitting Gap: 0.05424755126333802

================ FOLD 2 ================
RMSE: 0.04221872288161912
Train R2: 0.9429170120718128
Validation R2: 0.9102848590283902
Overfitting Gap: 0.032632153043422596

================ FOLD 3 ================
RMSE: 0.038423853624276166
Train R2: 0.9561841525137742
Validation R2: 0.9160741827207652
Overfitting Gap: 0.04010996979300896

================ FOLD 4 ================
RMSE: 0.043283832589358494
Train R2: 0.936724195910523
Validation R2: 0.9180687913933475
Overfitting Gap: 0.01865540451717551

================ FOLD 5 ================
RMSE: 0.04067244205823844
Train R2: 0.9537990997273493
Validation R2: 0.9384183257616879
Overfitting Gap: 0.015380773965661398

================ CATBOOST FINAL RESULTS ================
Average RMSE: 0.03996365938735264
Average Train R2: 0.9507554864489492
Average Validation R2: 0.91

In [38]:
print(oof_lgb.shape)
print(oof_xgb.shape)
print(oof_cat.shape)

print(test_lgb.shape)
print(test_xgb.shape)
print(test_cat.shape)

(77299,)
(77299,)
(77299,)
(41778,)
(41778,)
(41778,)


In [39]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ==========================================
# SIMPLE WEIGHTED ENSEMBLE
# ==========================================

ensemble_oof = (
    0.3 * oof_lgb +
    0.2 * oof_xgb +
    0.5 * oof_cat
)

# ==========================================
# EVALUATE ENSEMBLE
# ==========================================

ensemble_rmse = np.sqrt(mean_squared_error(y, ensemble_oof))
ensemble_r2 = r2_score(y, ensemble_oof)

print("Ensemble RMSE:", ensemble_rmse)
print("Ensemble R2:", ensemble_r2)

Ensemble RMSE: 0.04020217861537028
Ensemble R2: 0.9200601236763654


In [40]:
best_rmse = 999
best_weights = None

for lgb_w in np.arange(0.0, 1.1, 0.1):

    for xgb_w in np.arange(0.0, 1.1, 0.1):

        for cat_w in np.arange(0.0, 1.1, 0.1):

            total = lgb_w + xgb_w + cat_w

            if total == 0:
                continue

            # normalize
            lgb = lgb_w / total
            xgb = xgb_w / total
            cat = cat_w / total

            ensemble_oof = (
                lgb * oof_lgb +
                xgb * oof_xgb +
                cat * oof_cat
            )

            rmse = np.sqrt(mean_squared_error(y, ensemble_oof))

            if rmse < best_rmse:

                best_rmse = rmse

                best_weights = (lgb, xgb, cat)

print("Best RMSE:", best_rmse)

print("Best Weights:")
print("LGBM:", best_weights[0])
print("XGB :", best_weights[1])
print("CAT :", best_weights[2])

Best RMSE: 0.03990663850775392
Best Weights:
LGBM: 0.0
XGB : 0.2
CAT : 0.8


From this we can conclude that our catboost model is working well and LGBM has no weightage here ..
* LGBM predictions are probably highly correlated
* LGBM may actually be hurting ensemble 
* Here we will go for solo Catboost model as  LightGBM is weaker and Xgboost is slightly overfitting  

In [41]:
from catboost import CatBoostRegressor

final_cat_model = CatBoostRegressor(
    iterations=2500,
    learning_rate=0.03,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=200
)

final_cat_model.fit(X, y)

0:	learn: 0.1385237	total: 10ms	remaining: 25s
200:	learn: 0.0391612	total: 2.01s	remaining: 23s
400:	learn: 0.0360119	total: 3.96s	remaining: 20.7s
600:	learn: 0.0341128	total: 5.82s	remaining: 18.4s
800:	learn: 0.0328632	total: 7.67s	remaining: 16.3s
1000:	learn: 0.0318589	total: 9.71s	remaining: 14.5s
1200:	learn: 0.0310829	total: 11.8s	remaining: 12.8s
1400:	learn: 0.0304465	total: 13.6s	remaining: 10.6s
1600:	learn: 0.0299019	total: 15.4s	remaining: 8.66s
1800:	learn: 0.0294298	total: 17.2s	remaining: 6.66s
2000:	learn: 0.0290547	total: 19.3s	remaining: 4.82s
2200:	learn: 0.0286725	total: 22s	remaining: 2.99s
2400:	learn: 0.0283135	total: 24.9s	remaining: 1.03s
2499:	learn: 0.0281558	total: 26.1s	remaining: 0us


CatBoostRegressor(depth=6, eval_metric='RMSE', iterations=2500, learning_rate=0.03, loss_function='RMSE', random_seed=42, verbose=200)

In [42]:
final_predictions = final_cat_model.predict(X_test)

In [46]:
submission = pd.DataFrame({
    "Index": X_test.index,
    "demand": final_predictions
})

submission.to_csv("final_submission.csv", index=False)

print(submission.shape)
print(submission.head())

(41778, 2)
   Index    demand
0      0  0.061748
1      1  0.044412
2      2  0.036026
3      3  0.045621
4      4  0.048301
